# Phase 2: Sync Agent Migration — Production Code Tests

Testing LerimOAIAgent.sync() end-to-end:
- Prompt builder produces correct instructions
- Agent initializes with proxy detection
- Full sync flow with real session trace

In [ ]:
import os, sys, json, tempfile
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio

nest_asyncio.apply()

load_dotenv(Path("/Users/kargarisaac/codes/personal/lerim/lerim-cli/.env"))
sys.path.insert(0, str(Path("/Users/kargarisaac/codes/personal/lerim/lerim-cli/.claude/worktrees/agent-af7d097c/src")))

# Disable OAI SDK tracing (it tries to export to OpenAI servers using OPENAI_API_KEY)
from agents import set_tracing_disabled
set_tracing_disabled(disabled=True)

from lerim.runtime.oai_agent import LerimOAIAgent
from lerim.runtime.prompts.oai_sync import build_oai_sync_prompt
from lerim.config.settings import get_config

print("All imports successful (tracing disabled)")
cfg = get_config()
print(f"Provider: {cfg.lead_role.provider}/{cfg.lead_role.model}")

## Test 1 — Prompt Builder

In [ ]:
# Test prompt builder: build a sync prompt with test paths, print it, verify structure.
from lerim.runtime.agent import _build_artifact_paths

test_trace = Path("/tmp/lerim-test/trace.jsonl")
test_memory = Path("/tmp/lerim-test/memory")
test_run = Path("/tmp/lerim-test/workspace/sync-test-001")
test_artifacts = _build_artifact_paths(test_run)
test_metadata = {
	"run_id": "sync-test-001",
	"trace_path": str(test_trace),
	"repo_name": "lerim-cli",
}

prompt = build_oai_sync_prompt(
	trace_file=test_trace,
	memory_root=test_memory,
	run_folder=test_run,
	artifact_paths=test_artifacts,
	metadata=test_metadata,
)

print(f"Prompt length: {len(prompt)} chars")
print("---")
print(prompt)
print("---")

# Verify structure: must contain key paths and steps
assert str(test_trace) in prompt, "trace_path missing from prompt"
assert str(test_memory) in prompt, "memory_root missing from prompt"
assert str(test_run) in prompt, "run_folder missing from prompt"
assert "extract_pipeline" in prompt, "extract_pipeline step missing"
assert "summarize_pipeline" in prompt, "summarize_pipeline step missing"
assert "write_memory" in prompt, "write_memory step missing"
assert "memory_actions" in prompt, "memory_actions report reference missing"
assert "SCAN EXISTING MEMORIES" in prompt, "step 1 heading missing"
assert "WRITE REPORT" in prompt, "step 6 heading missing"
print("\nAll prompt structure assertions passed")

## Test 2 — Agent Initialization

In [ ]:
# Test agent initialization: create LerimOAIAgent, verify model, proxy, config.
agent = LerimOAIAgent(config=cfg)

print(f"Agent config provider: {agent.config.lead_role.provider}")
print(f"Agent config model:    {agent.config.lead_role.model}")
print(f"Needs proxy:           {agent._needs_proxy}")
print(f"Proxy object:          {type(agent._proxy).__name__ if agent._proxy else 'None'}")
print(f"Lead model type:       {type(agent._lead_model).__name__}")
print(f"Codex opts keys:       {list(agent._codex_opts.keys())}")
print(f"Thread opts keys:      {list(agent._thread_opts.keys())}")

# Verify model is a LitellmModel instance
from agents.extensions.models.litellm_model import LitellmModel
assert isinstance(agent._lead_model, LitellmModel), "Lead model is not a LitellmModel"

# Verify thread_opts has required fields
assert "model" in agent._thread_opts, "model missing from thread_opts"
assert "approval_policy" in agent._thread_opts, "approval_policy missing"
assert agent._thread_opts["approval_policy"] == "never", "approval_policy should be 'never'"

# Verify proxy is configured correctly based on provider
provider = cfg.lead_role.provider.strip().lower()
if provider in ("openrouter", "openai"):
	assert not agent._needs_proxy, f"Provider {provider} should not need proxy"
else:
	assert agent._needs_proxy, f"Provider {provider} should need proxy"
	assert agent._proxy is not None, "Proxy object should be created"

print("\nAll agent init assertions passed")

## Test 3 — Full Sync with Mock Trace

**NOTE:** This makes REAL LLM calls (MiniMax) so it costs tokens and takes time.

In [ ]:
import tempfile
import logging

# Enable logging so we can see the agent's progress
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

# Create temp dirs for memory_root and workspace_root
tmp_base = Path(tempfile.mkdtemp(prefix="lerim-phase2-"))
memory_root = tmp_base / "memory"
workspace_root = tmp_base / "workspace"
(memory_root / "decisions").mkdir(parents=True)
(memory_root / "learnings").mkdir(parents=True)
(memory_root / "summaries").mkdir(parents=True)
workspace_root.mkdir(parents=True)

# Write mock trace file
trace_path = tmp_base / "trace.jsonl"
mock_messages = [
	{"role": "user", "content": "Let's use PostgreSQL for the database. It's better for our scale."},
	{"role": "assistant", "content": "Good choice. PostgreSQL handles concurrent writes well and has excellent JSONB support for our session data."},
	{"role": "user", "content": "Also, I learned that we should always set statement_timeout to prevent long-running queries from blocking the connection pool."},
	{"role": "assistant", "content": "That's a great insight. Setting statement_timeout (e.g., 30s) prevents query pile-ups. I'll add it to the database config."},
]
with open(trace_path, "w") as f:
	for msg in mock_messages:
		f.write(json.dumps(msg) + "\n")

print(f"Temp base:      {tmp_base}")
print(f"Memory root:    {memory_root}")
print(f"Workspace root: {workspace_root}")
print(f"Trace path:     {trace_path}")
print(f"Trace lines:    {len(mock_messages)}")
print()

# Create agent and run sync
sync_agent = LerimOAIAgent(default_cwd=str(tmp_base), config=cfg)
print("Running sync (this makes real LLM calls)...")
print("=" * 60)

result = sync_agent.sync(
	trace_path=trace_path,
	memory_root=memory_root,
	workspace_root=workspace_root,
)

print("=" * 60)
print("\nSync completed!")
print(f"Cost: ${result.get('cost_usd', 0):.4f}")
print(f"Counts: {json.dumps(result.get('counts', {}), indent=2)}")
print(f"Written paths: {result.get('written_memory_paths', [])}")
print(f"Summary path: {result.get('summary_path', '')}")
print(f"Run folder: {result.get('run_folder', '')}")

# Verify memories were created
decision_files = list((memory_root / "decisions").rglob("*.md"))
learning_files = list((memory_root / "learnings").rglob("*.md"))
summary_files = list((memory_root / "summaries").rglob("*.md"))
print(f"\nDecision files:  {len(decision_files)}")
print(f"Learning files:  {len(learning_files)}")
print(f"Summary files:   {len(summary_files)}")

total_memories = len(decision_files) + len(learning_files)
assert total_memories > 0, "No memory files were written!"
print(f"\nTotal memories written: {total_memories} -- OK")

# Read and print each memory file
print("\n" + "=" * 60)
print("MEMORY FILE CONTENTS")
print("=" * 60)
for f in decision_files + learning_files:
	print(f"\n--- {f.relative_to(memory_root)} ---")
	print(f.read_text(encoding="utf-8"))

# Read and print the memory_actions.json report
actions_path = Path(result["artifacts"]["memory_actions"])
print("\n" + "=" * 60)
print("MEMORY ACTIONS REPORT")
print("=" * 60)
actions_data = json.loads(actions_path.read_text(encoding="utf-8"))
print(json.dumps(actions_data, indent=2))

## Summary

In [ ]:
print("Phase 2 Test Results")
print("=" * 40)
print(f"1. Prompt builder:    PASS")
print(f"2. Agent init:        PASS")
if 'result' in dir():
	counts = result.get("counts", {})
	total = counts.get("add", 0) + counts.get("update", 0)
	print(f"3. Full sync:         PASS ({total} memories written, ${result.get('cost_usd', 0):.4f})")
else:
	print(f"3. Full sync:         SKIPPED (cell not run)")
print()
print("Temp directory (inspect manually if needed):")
if 'tmp_base' in dir():
	print(f"  {tmp_base}")